# Homework 2: Anomaly Detection

In [ ]:
import numpy as np

import torch
from torch import nn
import torchvision

import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

SEED = 123456789
rng = np.random.default_rng(SEED)

_ = torch.manual_seed(SEED + 1)

## Dataset

In this homework, we are working with the following datasets:
- [MNIST dataset of handwritten digits](http://yann.lecun.com/exdb/mnist/);
- [EMNIST extension of MNIST](https://www.nist.gov/itl/products-and-services/emnist-dataset);
- [Fashion MNIST](https://github.com/zalandoresearch/fashion-mnist).

We want to build an anomaly detection algorithm for handwirtten digits:
- train split of MNIST is used as the training sample of normal data;
- test splits of MNIST and EMNIST's 'digits' is used as test samples of normal data;
- test splits of FashionMNIST and EMNIST's 'letters' is used as test samples of anomalious data.

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
])

normal_train = torchvision.datasets.MNIST(root='data/', train=True, download=True, transform=transform)
normal_train_dataloader = torch.utils.data.DataLoader(normal_train, shuffle=True, batch_size=32)

### Test datasets

normal_test = torchvision.datasets.MNIST(root='data/', train=False, download=True, transform=transform)
normal_test_dataloader = torch.utils.data.DataLoader(normal_test, shuffle=False, batch_size=32)

anomalies_fashion = torchvision.datasets.FashionMNIST(root='data/', train=False, download=True, transform=transform)
anomalies_fashion_dataloader = torch.utils.data.DataLoader(anomalies_fashion, shuffle=False, batch_size=32)

### note: EMNIST images are transposed
transform_emnist = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Lambda(lambda x: torch.transpose(x, 1, 2))
])

anomalies_letters = torchvision.datasets.EMNIST(root='data/', split='letters', download=True, train=False, transform=transform_emnist)
anomalies_letter_dataloader = torch.utils.data.DataLoader(anomalies_letters, shuffle=False, batch_size=32)

normal_digits = torchvision.datasets.EMNIST(root='data/', split='digits', download=True, train=False, transform=transform_emnist)
normal_digits_dataloader = torch.utils.data.DataLoader(normal_digits, shuffle=False, batch_size=32)

## Training

### Task 1. Implement AE (0.25 points)

Implement AutoEncoder in torch. Consult torch documentation if neccessary.
You are also free to use TensorFlow or Jax if you are more familiar with these frameworks.

Both encoder and decoder networks must have more than 1 layer (3+ layer are recommended, nonlinearities are not counted). Adjust the number of units so you can train the network for at least 8 epochs on you computer within a reasonable amount of time.

In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self, latent: int):
        """
        latent: dimensionality of the latent space.
        """
        super(AutoEncoder, self).__init__()
        
        self.latent = latent

        self.encoder = ...
        self.decoder = ...

    def encode(self, x):
        x = torch.flatten(x, start_dim=1)
        return self.encoder(x)

    def decode(self, z):
        x_reco = self.decoder(z)
        return torch.reshape(x_reco, shape=(x_reco.shape[0], 1, 28, 28))

    def forward(self, x):
        return self.decode(self.encode(x))

### Task 2. Train AE (0.25 points)

Train you model as an ordinary AE. Train it for at least 8 epochs (32 or more is recommended).
Use AdamW optimizer with learning rate `1.0e-3` as the starting point and other parameters set to default values. Adjust the learning rate if neccessary. Adjust the number of latent dimensions if neccessary.

In [ ]:
ae = AutoEncoder(10).to(device)
optimizer = torch.optim.AdamW(params=ae.parameters(), lr=1.0e-3, )

In [ ]:
n_epochs = 8
n_iterations = len(normal_train_dataloader)

losses = np.ndarray(shape=(n_epochs, n_iterations))

ae.train()

for i in tqdm(range(n_epochs)):
    for j, (X_batch, _) in enumerate(normal_train_dataloader):
        X_batch = X_batch.to(device)

        optimizer.zero_grad()

        X_reco = ...
        loss = ...

        losses[i, j] = loss.item()

        loss.backward()
        optimizer.step()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(np.mean(losses, axis=1))

### Visual evaluation

We display a sample of original and reconstructed images. Such diagnostic plots can help undentify whether AE was improperly trained.

In [ ]:
ae.eval()

X_sample, _ = next(iter(normal_train_dataloader))
X_reco = ae(X_sample.to(device)).cpu().detach().numpy()
X_sample = X_sample.cpu().detach().numpy()

n = 4
fig = plt.figure(figsize=(2 * 3 * n, 3 * n))
axes = fig.subplots(n, 2 * n, squeeze=False).ravel()

for i in range(n * n):
    axes[2 * i].imshow(X_sample[i, 0], cmap=plt.cm.gray_r, vmin=0.0, vmax=1.0)
    axes[2 * i + 1].imshow(X_reco[i, 0], cmap=plt.cm.gray_r, vmin=0.0, vmax=1.0)

## Test

### Visual evaluation

Following the example above, display original and reconstructed images for each of test datasets.

In [ ]:
# ...

### Numerical evaluation

Implement a scoring function that receives a batch of sample images and outputs an anomaly score. The higher the score, the higher the algorithm’s confidence that the sample belongs to the anomalous class.

In [ ]:
def score(model, X):
    ...

In [ ]:
ae.eval()

scores_mnist = np.concatenate([
    score(ae, X_batch.to(device)).detach().cpu().numpy()
    for X_batch, _ in tqdm(normal_test_dataloader, desc='MNIST')
])

scores_fashion = np.concatenate([
    score(ae, X_batch.to(device)).detach().cpu().numpy()
    for X_batch, _ in tqdm(anomalies_fashion_dataloader, desc='Fashion')
])

scores_letters = np.concatenate([
    score(ae, X_batch.to(device)).detach().cpu().numpy()
    for X_batch, _ in tqdm(anomalies_letter_dataloader, desc='Letters')
])

scores_digits = np.concatenate([
    score(ae, X_batch.to(device)).detach().cpu().numpy()
    for X_batch, _ in tqdm(normal_digits_dataloader, desc='Digits')
])

In [ ]:
scores = np.concatenate([scores_mnist, scores_fashion, scores_letters, scores_digits])
labels = np.concatenate([
    np.ones(scores_mnist.shape[0]),
    np.zeros(scores_fashion.shape[0]),
    np.zeros(scores_letters.shape[0]),
    np.ones(scores_digits.shape[0]),
])

### ROC curve

Compute ROC curve and ROC AUC.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

### we flip the labels because the lower the score the more probable is the normal class, which has y = 1
fpr, tpr, thresholds = roc_curve(1 - labels, scores)
auc = roc_auc_score(1 - labels, scores)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", color='black')  # random classifier baseline

plt.xlim([0, 1])
plt.ylim([0, 1])

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()

### Task 3. Anomaly Detection (0.25 points)

On the training set of normal samples, compute the threshold $\tau$ that captures 99% of the normal samples:
$$P\left(\mathrm{score}(x) < \tau \mid x \text{ is normal}\right) = 0.99$$

For each test dataset, compute False Negative Rate (FNR) for normal datasets, and False Positive Rate (FPR) for anomalious datasets.

In [ ]:
# ...

### Task 4. Denoising AutoEncoder (0.25 points)

Repeat the workflow for Denoising Auto-Encoder (DAE):
- create a new model with the same architecture as the one used for ordinary AE;
- rain the DAE using a denoising setup, applying Gaussian noise with a standard deviation of `std = 0.1`;
- repeat all evaluation steps for the DAE.

Adjust `std` of the Gaussian noise if neccessary.

In [ ]:
# ...

### Task 5. Robust Denoising AutoEncoder (extra, optional, 0.5 points)

Implement and evaluate a Robust Denoising Auto-Encoder. Instead of the denoising reconstruction loss, use the unrolled version $\mathcal{L}$:

\begin{align}
    X_0 &\sim \mathrm{data};\\
    X_{t + 1} &= \mathrm{AE}(X_t + \varepsilon_t);\\
    \mathcal{L} &= \left\| X_n - X_0 \right\|^2;
\end{align}
where $\varepsilon_t$ is Gaussian noise (as in DAE). Train progressively increasing $n$ up to 8, e.g., 4 epochs with $n = 1$ (standard DAE), then 4 epochs with $n = 2$ etc.

### Task 6. Residual Denoising AutoEncoder (extra, optional, 0.5 points)

Implement and evaluate a Denoising Auto-Encoder with residual connections (rDAE): instead of returning a reconstructed image, the decoder outputs the difference between the input (noisy sample) and the reconstructed image:

\begin{align}
    \mathrm{denoise}(\chi) &= \chi + \mathrm{rDAE}(\chi);\\
    \mathcal{L} &= \| X - \mathrm{denoise}(X + \varepsilon) \|^2
\end{align}

where $\varepsilon$ is Gaussian noise (as in DAE).

Note: this architecture is no longer technically an AutoEncoder.

## Generative AE (optional, no points, just for fun)

Using a trained DAE (Robust DAE or Residual DAE for better results), implement Markov Chain:
- take a batch of MNIST images $X_0$;
- iterate for $n$ steps (recommended 32+) the following transformation: $X_{t + 1} = AE(X_t + \varepsilon)$, where $\varepsilon$ is Gaussian noise with the same `std` as was used during training (resample noise at each iteration);
- visualize the original $X_0$ and the final $X_n$.

Repeat using uniform noise as the initial sample.

In [ ]:
# ...